# Inkling 2 Mini on Kaggle with Strata

Run Inkling 2 Mini (SQ-10) on Kaggle with a public API protected by an API key.

Set the accelerator to GPU T4 x2 and turn Internet on before you start.

## 1. Setup

### Install Strata and cloudflared
Run once.

In [ ]:
%%bash
nvidia-smi
curl -fsSL https://github.com/Niko1221/Strata/releases/latest/download/install.sh -o /tmp/install.sh
sh /tmp/install.sh
curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
export PATH="$HOME/.local/bin:$PATH"
strata --version
cloudflared --version

### Start the model
Replace `YOUR_SECRET_KEY` with your own key, then run once. The public address is printed in the output.

To try another variant, change `MODEL=sq-10` in the cell below.

| Name | Size | Notes |
|---|---|---|
| `sq-10` | 10.47 GB | Default. Fits one T4. |
| `sq-15` | 14.35 GB | Better quality. Tight on one T4. |
| `sq-20` | 19.35 GB | Best quality. Needs both T4s. |

The `ud-*` variants (Unsloth files) currently fail Strata's file size check, so stick to the `sq-*` ones.

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
MODEL=sq-10
strata launch --family inkling-2-mini --model $MODEL --yes --no-tui --port 8080 \
  --access cloudflare --api-key YOUR_SECRET_KEY

## 2. Test

### Test the model
Use the same key as above. Shows the reply and the speed.

In [ ]:
import requests, json, time

KEY = "YOUR_SECRET_KEY"
prompt = "Say hello in one short sentence."

headers = {"Authorization": f"Bearer {KEY}"}

try:
    model = requests.get("http://127.0.0.1:8080/v1/models", headers=headers, timeout=5).json()["data"][0]["id"]
except Exception:
    model = "unknown"
print(f"model: {model}\n")

start = time.time()
tokens = 0
r = requests.post("http://127.0.0.1:8080/v1/chat/completions", headers=headers,
    json={"messages": [{"role": "user", "content": prompt}], "max_tokens": 256, "stream": True},
    stream=True, timeout=300)

for line in r.iter_lines():
    if not line or not line.startswith(b"data: "):
        continue
    chunk = line[6:]
    if chunk == b"[DONE]":
        break
    delta = json.loads(chunk)["choices"][0]["delta"].get("content") or ""
    if delta:
        tokens += 1
        print(delta, end="", flush=True)

elapsed = time.time() - start
print(f"\n\n{tokens} tokens in {elapsed:.1f}s = {tokens/elapsed:.1f} tok/s")

## 3. Public access

### Show the address and key
Run if you lost the public address or the key.

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
strata launch --status
strata launch --show-key

### Call it from anywhere
Use this curl command from any computer. Replace the address and the key.

```bash
curl https://YOUR-TUNNEL-URL/v1/chat/completions \
  -H "Authorization: Bearer YOUR_SECRET_KEY" \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Say hello in one short sentence."}], "max_tokens": 256}'
```

## 4. Stop

### Stop the model
Run when you are done, or before starting a different variant.

In [ ]:
%%bash
pkill -9 -f cloudflared; pkill -9 -f strata; sleep 3; echo stopped